# 🚀 Qwen-2.5-0.5B Sparse Upcycling: OS-MoE, GOR-v2 & GEO-Muon Benchmark
Notebook này hướng dẫn quy trình **Sparse Upcycling** chuyển đổi mô hình thương mại **Qwen-2.5-0.5B** thành **Qwen-OS-MoE-0.5B**:
1. **Cơ chế GOR-v2:** Phá vỡ bão hòa độ sâu bằng phép chiếu trực giao $u^\perp$.
2. **Cơ chế OS-MoE:** Phân rã FFN thành 1 Shared Expert + 8 Sub-Experts với Cosine-Bounded Router (giảm ~43% FLOPs).
3. **Tối ưu hóa GEO-Muon:** Tối ưu hóa ma trận 2D trên đa tạp Stiefel bằng phép lặp Newton-Schulz bậc 5.
4. **Bộ 4 Metric Chuẩn 2026:** So sánh trực tiếp với các mô hình Frontier (DeepSeekMoE, Moonlight, Qwen, LLaMA).

## 1. Thiết lập Môi trường & Kiểm tra GPU

In [ ]:
!pip install -q transformers accelerate datasets matplotlib torch einops
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Tải Mã Nguồn Upcycling & Thư Viện Chẩn Đoán

In [ ]:
import os
if not os.path.exists("qwen_moefication.py"):
    !git clone https://github.com/khangkaka066/Alba-model.git
    %cd Alba-model/qwen_upcycling
import sys
sys.path.append(".")
from qwen_moefication import upcycle_qwen_to_osmoe
from qwen_geo_muon import create_qwen_geomuon_optimizer
from qwen_eval_metrics import evaluate_effective_depth, evaluate_routing_entropy, evaluate_spectral_isometry, plot_qwen_comparison_dashboard

## 3. Bước 1: Đánh Giá Nền Tảng (Baseline) Trên Qwen-2.5-0.5B Gốc
Đo lường độ sâu hiệu dụng ban đầu $\mathcal{D}_{\text{eff}}$ qua 24 tầng bằng Linear CKA.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_orig = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

test_corpus = [
    "The quick brown fox jumps over the lazy dog. Mathematical formalization guarantees convergence.",
    "In deep transformer architectures, linear attention is provably incapable of extreme representation renewal.",
    "Mixture of experts enables sparse execution where each token routes to a dynamic subset of specialized sub-networks.",
    "Geometric manifold optimization projects gradients onto the Stiefel manifold to preserve spectral isometry."
]

print("Đang đo lường chỉ số nền tảng của Qwen-2.5-0.5B gốc...")
orig_depth = evaluate_effective_depth(model_orig, tokenizer, test_corpus)
orig_spectral = evaluate_spectral_isometry(model_orig)

orig_metrics = {
    "n_layers": orig_depth["n_layers"],
    "effective_depth": orig_depth["effective_depth"],
    "depth_ratio": orig_depth["depth_ratio"],
    "renewals": orig_depth["renewals"],
    "mean_condition_number": orig_spectral["mean_condition_number"]
}

print(f"Qwen-2.5-0.5B Gốc -> Effective Depth: {orig_metrics['effective_depth']:.2f} / {orig_metrics['n_layers']} (Tỷ lệ: {orig_metrics['depth_ratio']*100:.1f}%) | Kappa: {orig_metrics['mean_condition_number']:.1f}")

## 4. Bước 2: Thực Hiện Sparse Upcycling (Chuyển Thành Qwen-OS-MoE-0.5B + GOR-v2)
Tách toàn bộ 24 khối FFN thành 1 Shared Expert + 8 Sub-Experts, gắn Cosine-Bounded Router và bọc GOR-v2 Residual.

In [ ]:
print("Đang áp dụng Sparse Upcycling lên mô hình...")
model_upcycled = upcycle_qwen_to_osmoe(model_orig, num_experts=8, top_k=2, enable_gor=True)
print("Sparse Upcycling hoàn tất! Đã tích hợp thành công OS-MoE và GOR-v2.")

## 5. Bước 3: Huấn Luyện Thích Ứng (Continual Pretraining) Bằng GEO-Muon

In [ ]:
import torch.nn.functional as F

optimizer = create_qwen_geomuon_optimizer(model_upcycled, lr=2e-4)
model_upcycled.train()

steps = 50
print(f"Bắt đầu huấn luyện GEO-Muon ({steps} steps)...")
for s in range(1, steps + 1):
    optimizer.zero_grad()
    batch_text = [test_corpus[s % len(test_corpus)] for _ in range(2)]
    inputs = tokenizer(batch_text, return_tensors="pt", max_length=256, padding="max_length", truncation=True)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    inputs["labels"] = inputs["input_ids"].clone()
    
    outputs = model_upcycled(**inputs)
    lm_loss = outputs.loss
    
    ortho_loss = torch.tensor(0.0, device=lm_loss.device)
    routers = 0
    for m in model_upcycled.modules():
        if m.__class__.__name__ == "CosineBoundedRouter":
            gate_norm = F.normalize(m.gate, p=2, dim=-1, eps=1e-7)
            eye = torch.eye(m.num_experts, device=gate_norm.device, dtype=gate_norm.dtype)
            ortho_loss += torch.sum((torch.matmul(gate_norm, gate_norm.t()) - eye)**2) / (m.num_experts * (m.num_experts - 1))
            routers += 1
    if routers > 0:
        ortho_loss = ortho_loss / routers
        
    total_loss = lm_loss + 0.01 * ortho_loss
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model_upcycled.parameters(), 1.0)
    optimizer.step()
    
    if s % 10 == 0 or s == steps:
        print(f"Step {s:02d}/{steps} | LM Loss: {lm_loss.item():.4f} | Ortho Loss: {ortho_loss.item():.4f}")

## 6. Bước 4: Đo Lường Toàn Diện 4 Trụ Cột Trên Qwen-OS-MoE-0.5B Đã Upcycle

In [ ]:
print("Đang đo lường các chỉ số của Qwen-OS-MoE-0.5B...")
up_depth = evaluate_effective_depth(model_upcycled, tokenizer, test_corpus)
up_moe = evaluate_routing_entropy(model_upcycled, tokenizer, test_corpus)
up_spectral = evaluate_spectral_isometry(model_upcycled)

upcycled_metrics = {
    "n_layers": up_depth["n_layers"],
    "effective_depth": up_depth["effective_depth"],
    "depth_ratio": up_depth["depth_ratio"],
    "renewals": up_depth["renewals"],
    "has_moe": up_moe["has_moe"],
    "mean_entropy": up_moe["mean_entropy"],
    "load_cv": up_moe["load_cv"],
    "layer_entropies": up_moe["layer_entropies"],
    "mean_condition_number": up_spectral["mean_condition_number"]
}

print("=== KẾT QUẢ ĐỐI SÁNH ===")
print(f"Effective Depth: Qwen Gốc = {orig_metrics['effective_depth']:.2f} ({orig_metrics['depth_ratio']*100:.1f}%) -> Qwen-OS-MoE = {upcycled_metrics['effective_depth']:.2f} ({upcycled_metrics['depth_ratio']*100:.1f}%)")
print(f"Routing Entropy: {upcycled_metrics['mean_entropy']:.2f} / 2.08 (Load CV: {upcycled_metrics['load_cv']:.3f})")
print(f"Spectral Condition Number: Qwen Gốc = {orig_metrics['mean_condition_number']:.1f} -> Upcycled GEO-Muon = {upcycled_metrics['mean_condition_number']:.1f}")

## 7. Bước 5: Kết Xuất Biểu Đồ Đối Sánh Trực Quan Với Các Mô Hình Frontier

In [ ]:
from IPython.display import Image, display

dashboard_output = "qwen_upcycling_dashboard.png"
plot_qwen_comparison_dashboard(orig_metrics, upcycled_metrics, dashboard_output)
display(Image(dashboard_output))